# License
This notebook is licensed under the MIT License. See the LICENSE file in the repository root for details.

In [ ]:
# Setup for Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Packages installed.')


# Survey Data Cleaning: An In-Depth Guide

Real-world survey data is rarely clean. It contains duplicates, inconsistent spelling, outliers, and missing values. This notebook demonstrates a professional workflow for cleaning a messy dataset using `pandas`.

## Learning Objectives
1.  **Detect and remove duplicates.**
2.  **Standardize categorical variables** (e.g., mapping "m", "Male", "M" to "Male").
3.  **Handle missing data** effectively (Drop vs Impute).
4.  **Detect outliers** in numeric responses.
5.  **Recode reverse-coded items** (common in psychological surveys).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set seed for reproducibility
np.random.seed(42)

## 1. Generating a "Dirty" Dataset
We will simulate a dataset of 500 respondents with intentional errors to practice cleaning.

In [ ]:
def generate_messy_data(n=500):
    ids = range(1, n + 1)
    
    # Gender with inconsistencies
    genders_opts = ['Male', 'Female', 'Non-binary', 'm', 'f', 'M', 'F', 'male', 'female', np.nan]
    genders = np.random.choice(genders_opts, n, p=[0.3, 0.3, 0.05, 0.1, 0.1, 0.05, 0.05, 0.02, 0.02, 0.01])
    
    # Age with outliers and missing
    ages = np.random.normal(35, 10, n)
    ages[0] = 150 # Impossible age
    ages[1] = 5   # Child age in adult survey
    ages[np.random.choice(n, 20, replace=False)] = np.nan
    
    # Likert Scale (1-5) with string errors
    sat_opts = [1, 2, 3, 4, 5, '3', '4', np.nan, 99] # 99 is a common code for 'Refused'
    satisfaction = np.random.choice(sat_opts, n)
    
    # Reverse Coded Item (1-5)
    # Q: "I often feel stressed" (High score = Bad)
    stress = np.random.randint(1, 6, n)
    
    df = pd.DataFrame({
        'ID': ids,
        'Gender': genders,
        'Age': ages,
        'Satisfaction': satisfaction,
        'Stress_Level': stress
    })
    
    # Introduce duplicates
    df = pd.concat([df, df.iloc[:10]], ignore_index=True)
    
    return df

raw_df = generate_messy_data()
print(f"Original Shape: {raw_df.shape}")
print(raw_df.head(10))

## 2. Removing Duplicates
Duplicate entries can skew results. We check for full row duplicates.

In [ ]:
print(f"Duplicates before: {raw_df.duplicated().sum()}")

df_clean = raw_df.drop_duplicates()
print(f"Shape after dropping duplicates: {df_clean.shape}")

## 3. Standardizing Categorical Data
The 'Gender' column has multiple representations for the same category (e.g., 'm', 'M', 'Male'). We map them to a standard format.

In [ ]:
print("Unique values before cleaning:", df_clean['Gender'].unique())

# 1. Convert to title case (handle 'male', 'female')
df_clean['Gender'] = df_clean['Gender'].str.title()

# 2. Map abbreviations
gender_map = {
    'M': 'Male',
    'F': 'Female',
}
df_clean['Gender'] = df_clean['Gender'].replace(gender_map)

print("Unique values after cleaning:", df_clean['Gender'].unique())

# Check distribution
print(df_clean['Gender'].value_counts(dropna=False))

## 4. Cleaning Numeric Data
The 'Satisfaction' column contains strings ('3') and codes (99). We need to convert it to numeric and handle invalid values.

In [ ]:
# Coerce to numeric (turns strings like '3' to 3, and errors to NaN)
df_clean['Satisfaction'] = pd.to_numeric(df_clean['Satisfaction'], errors='coerce')

# Handle 99 (Refused) -> Replace with NaN or Drop
# We'll assume valid scale is 1-5
df_clean.loc[df_clean['Satisfaction'] > 5, 'Satisfaction'] = np.nan

print("Satisfaction distribution:")
print(df_clean['Satisfaction'].value_counts(dropna=False).sort_index())

## 5. Handling Outliers
The 'Age' column has impossible values (e.g., 150, 5). We visualize and cap them.

In [ ]:
plt.figure(figsize=(10, 4))
sns.boxplot(x=df_clean['Age'])
plt.title('Age Distribution (with outliers)')
plt.show()

# Define valid range (e.g., 18 to 100)
valid_age_mask = (df_clean['Age'] >= 18) & (df_clean['Age'] <= 100)

# Mark invalid ages as NaN
df_clean.loc[~valid_age_mask, 'Age'] = np.nan

print(f"Missing ages after cleaning outliers: {df_clean['Age'].isna().sum()}")

## 6. Missing Data Imputation
Strategies:
1.  **Drop**: If N is large and missingness is low.
2.  **Mean/Median**: For continuous vars (Age).
3.  **Mode**: For categorical vars.

In [ ]:
# Fill Age with Median (robust to skew)
median_age = df_clean['Age'].median()
df_clean['Age'].fillna(median_age, inplace=True)

# Drop rows where essential demographic info (Gender) is missing
df_clean.dropna(subset=['Gender'], inplace=True)

print(f"Final Dataset Shape: {df_clean.shape}")

## 7. Recoding Variables
Sometimes high scores mean "Bad" (e.g., Stress). For consistency, we might want to reverse this so High Score = Good (e.g., "Calmness").

Formula for 1-N scale: `New_Val = (Max + Min) - Old_Val`
For 1-5 scale: `6 - Val`

In [ ]:
# Recode Stress (1-5) to Calmness (1-5)
df_clean['Calmness'] = 6 - df_clean['Stress_Level']

print(df_clean[['Stress_Level', 'Calmness']].head())

## Summary
We have transformed a messy dataset into a clean one ready for analysis. Always report your cleaning steps in your methodology section!